# Homework 5 — Deploying Machine Learning Models

ML Zoomcamp 2026 · Module 5

Assignment: [cohorts/2026/homework/05-deployment](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/main/cohorts/2026/homework/05-deployment)

Goal: take the frozen 2026 lead-scoring model (`DictVectorizer` + `LogisticRegression`), load it with `pickle`, serve it with FastAPI/uvicorn and run the same prediction inside the reference Docker container.

Due: **19 Oct 2026, 23:00 UTC**

## Setup

In [ ]:
import hashlib
import json
import re
import subprocess
import time
import tomllib
from pathlib import Path
from urllib.request import Request, urlopen

RELEASE = Path('release')
BASE_URL = 'http://localhost:9696'

---
## Getting the release

The model, API, lockfile and Dockerfile are part of the course release. They are downloaded into `release/` (not committed — same as `data/` in earlier modules).

In [ ]:
RAW = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/homework/05-deployment'
FILES = [
    'pipeline.bin', 'model.py', 'predict.py', 'smoke_test.py', 'q6_test.py',
    'feature_defaults.json', 'model_metadata.json',
    'pyproject.toml', 'uv.lock', '.python-version', 'Dockerfile',
]

RELEASE.mkdir(exist_ok=True)
for name in FILES:
    if not (RELEASE / name).exists():
        subprocess.run(['curl', '-sSfL', '-o', str(RELEASE / name), f'{RAW}/{name}'], check=True)

sorted(p.name for p in RELEASE.iterdir())

Verify the artifact checksum before using it:

In [ ]:
EXPECTED_SHA256 = '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1'

model_sha256 = hashlib.sha256((RELEASE / 'pipeline.bin').read_bytes()).hexdigest()
metadata = json.loads((RELEASE / 'model_metadata.json').read_text())

print(model_sha256, ' pipeline.bin')
assert model_sha256 == EXPECTED_SHA256 == metadata['model_sha256']

Create the locked environment (Python 3.11 + pinned deps) and run the smoke test:

In [ ]:
!cd release && uv sync --locked -q && uv run python smoke_test.py

---
## Q1. Environment check

`uv --version` — setup check only, not graded.

In [ ]:
# Q1
!uv --version

---
## Q2. Locked dependency

Which Scikit-Learn version is part of the 2026 reference environment?

*Options: 1.6.1 / 1.7.2 / 1.8.0 / 2.0.0*

In [ ]:
# Q2
pyproject = tomllib.loads((RELEASE / 'pyproject.toml').read_text())
lock = tomllib.loads((RELEASE / 'uv.lock').read_text())

print('pyproject.toml:', [d for d in pyproject['project']['dependencies'] if d.startswith('scikit-learn')])
print('uv.lock       :', [p['version'] for p in lock['package'] if p['name'] == 'scikit-learn'])

---
## Q3. Load the model

Load `pipeline.bin` with `pickle` and call `predict_proba` for the first lead (see [`q3_predict.py`](q3_predict.py)). Run inside the locked env so the unpickled scikit-learn version matches the one used for training.

In [ ]:
# Q3
!cat q3_predict.py

In [ ]:
!cd release && uv run python ../q3_predict.py

---
## Q4. Serve the model

Start the provided API with uvicorn in the background, send the second lead ([`q4_test.py`](q4_test.py)), then stop the server.

In [ ]:
def wait_for_health(url=BASE_URL, timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urlopen(f'{url}/health') as r:
                return json.load(r)
        except Exception:
            time.sleep(0.5)
    raise TimeoutError(f'{url} did not become healthy')

In [ ]:
# Q4
server = subprocess.Popen(
    ['uv', 'run', 'uvicorn', 'predict:app', '--host', '0.0.0.0', '--port', '9696'],
    cwd=RELEASE, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
try:
    wait_for_health()
    !python q4_test.py
finally:
    server.terminate()
    server.wait()

---
## Q5. Container configuration

Which Python base-image tag is declared in the `Dockerfile`?

*Options: `python:3.9-slim-bullseye` / `python:3.11.15-slim-bookworm` / `python:3.12-slim-bookworm` / `python:3.13.10-slim-bookworm`*

In [ ]:
# Q5
dockerfile = (RELEASE / 'Dockerfile').read_text()
print(dockerfile)

base = re.search(r'^FROM\s+(\S+)', dockerfile, re.M).group(1)
print('base image tag:', base.split('@')[0])

---
## Q6. Run the container

```bash
docker build -t zoomcamp-model:2026-hw5 release/
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
```

Then send the same request as Q4 and check `/health` returns `status: ok` plus the checksum from `model_metadata.json`.

In [ ]:
!docker build -q -t zoomcamp-model:2026-hw5 release/

In [ ]:
# Q6
container = subprocess.run(
    ['docker', 'run', '--rm', '-d', '-p', '9696:9696', 'zoomcamp-model:2026-hw5'],
    check=True, capture_output=True, text=True,
).stdout.strip()
try:
    health = wait_for_health()
    assert health['status'] == 'ok'
    assert health['model_sha256'] == metadata['model_sha256']
    !python release/q6_test.py
    !python q4_test.py
finally:
    subprocess.run(['docker', 'stop', container], check=True, capture_output=True)

---
## Answers

| Question | Answer | Computed value |
|---|---|---|
| Q1 — `uv --version` | see cell above | setup check, not graded |
| Q2 — scikit-learn in lockfile | **1.7.2** | `scikit-learn==1.7.2` in `pyproject.toml` and `uv.lock` |
| Q3 — probability, lead 1 | **0.533** | 0.532994 |
| Q4 — API probability, lead 2 | **0.770** | 0.769799 (`conversion: true`) |
| Q5 — base image | **`python:3.11.15-slim-bookworm`** | pinned by digest in `Dockerfile` |
| Q6 — container probability, lead 2 | **0.770** | same artifact + same lockfile → same inference as Q4 |

Notes:

- The pickle must be loaded with the locked scikit-learn (1.7.2); the root project env uses a newer version and would warn on unpickling — hence `uv run` inside `release/`.
- The API fills missing numerics with training medians from `feature_defaults.json` and missing categoricals with `'NA'`; both test leads are complete, so no imputation kicks in.
- `/health` exposes the model checksum, which makes it easy to confirm the container serves the exact frozen artifact.